# PaySim — Mobile Money Transactions

| | |
|---|---|
| **Source** | Kaggle `ealaxi/paysim1` |
| **Origin** | Lopez-Rojas et al. — agent-based simulation calibrated on real African mobile-money logs |
| **License** | CC BY-SA 4.0 |
| **Samples** | 6,362,620 transactions over 744 simulated hours (30 days) |
| **Features** | step, type, amount, orig/dest account IDs + balances |
| **Labels** | `isFraud`, plus rule-based `isFlaggedFraud` |
| **Caveats** | Synthetic but calibrated on real logs; fraud only in TRANSFER + CASH_OUT. |
| **Production research suitability** | HIGH for transaction-sequence/behaviour modeling; account IDs enable per-customer history. |

In [1]:
import sys
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
from prep_utils import RAW, to_unified, dataset_report, numeric_summary, save_clean, save_unified_part

D = RAW / "financial" / "paysim"

In [2]:
import glob
csv = glob.glob(str(D / "*.csv"))[0]
df = pd.read_csv(csv)
print(df.shape)
df.head(3)

(6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0


## Cleaning + consistency checks

In [3]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"dropped {before - len(df)} duplicates; missing: {int(df.isna().sum().sum())}")
df["type"] = df["type"].astype("category")
assert df["isFraud"].isin([0, 1]).all()
# balance-equation violations are a known PaySim artifact — measure, keep as feature
orig_err = (df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"]).abs()
df["orig_balance_inconsistent"] = ((orig_err > 0.01) & (df["oldbalanceOrg"] > 0)).astype("int8")
print("orig balance inconsistency rate:", round(df["orig_balance_inconsistent"].mean(), 4))

dropped 0 duplicates; missing: 0


orig balance inconsistency rate: 0.4676


## Label verification
Fraud must occur only in TRANSFER/CASH_OUT per dataset design.

In [4]:
print(df.groupby("type", observed=True)["isFraud"].sum())
assert df.loc[df["isFraud"] == 1, "type"].isin(["TRANSFER", "CASH_OUT"]).all()
df["isFraud"].value_counts(normalize=True)

type
CASH_IN        0
CASH_OUT    4116
DEBIT          0
PAYMENT        0
TRANSFER    4097
Name: isFraud, dtype: int64


isFraud
0    0.998709
1    0.001291
Name: proportion, dtype: float64

## Timestamp normalization
`step` = 1 hour of simulation, 744 steps = 30 days. Anchor at 2023-01-01 UTC
(synthetic, flagged). Random within-hour jitter keeps sequence but breaks ties.

In [5]:
rng = np.random.default_rng(42)
anchor = pd.Timestamp("2023-01-01 00:00:00", tz="UTC")
df["event_time"] = anchor + pd.to_timedelta(df["step"] - 1, unit="h") \
    + pd.to_timedelta(rng.uniform(0, 3600, len(df)), unit="s")
df = df.sort_values("event_time").reset_index(drop=True)

## EDA

In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df["type"].value_counts().plot.bar(ax=axes[0], title="txn types")
df.groupby(df["step"] // 24)["isFraud"].mean().plot(ax=axes[1], title="fraud rate by day")
axes[2].hist(np.log1p(df["amount"]), bins=60); axes[2].set_title("log1p(amount)")
plt.tight_layout(); plt.savefig("../reports/paysim_eda.png", dpi=110); plt.show()

/tmp/ipykernel_179495/834098634.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("../reports/paysim_eda.png", dpi=110); plt.show()


In [7]:
numeric_summary(df, "paysim")

,count,mean,std,min,25%,50%,75%,max
step,6362620.0,2.433972e+02,1.423320e+02,1.0,156.00,239.000,3.350000e+02,7.430000e+02
amount,6362620.0,1.798619e+05,6.038582e+05,0.0,13389.57,74871.940,2.087215e+05,9.244552e+07
oldbalanceOrg,6362620.0,8.338831e+05,2.888243e+06,0.0,0.00,14208.000,1.073152e+05,5.958504e+07
newbalanceOrig,6362620.0,8.551137e+05,2.924049e+06,0.0,0.00,0.000,1.442584e+05,4.958504e+07
oldbalanceDest,6362620.0,1.100702e+06,3.399180e+06,0.0,0.00,132705.665,9.430367e+05,3.560159e+08
newbalanceDest,6362620.0,1.224996e+06,3.674129e+06,0.0,0.00,214661.440,1.111909e+06,3.561793e+08
isFraud,6362620.0,1.290820e-03,3.590480e-02,0.0,0.00,0.000,0.000000e+00,1.000000e+00
isFlaggedFraud,6362620.0,2.514687e-06,1.585775e-03,0.0,0.00,0.000,0.000000e+00,1.000000e+00
orig_balance_inconsistent,6362620.0,4.676154e-01,4.989502e-01,0.0,0.00,0.000,1.000000e+00,1.000000e+00


## Save clean + unified

In [8]:
save_clean(df, "paysim")
dataset_report(df, "paysim", label_col="isFraud",
               notes="Simulated 30-day window anchored 2023-01-01 (synthetic). Balance inconsistencies kept as feature.")

clean -> /home/suryaguru/StudioProjects/sentinel_fusion_ai/data/clean/paysim.parquet (6,362,620 rows)


report -> /home/suryaguru/StudioProjects/sentinel_fusion_ai/reports/paysim_stats.json


{'dataset': 'paysim',
 'rows': 6362620,
 'columns': 13,
 'memory_mb': 707.6,
 'duplicate_rows': 0,
 'missing_by_column': {},
 'dtypes': {'step': 'int64',
  'type': 'category',
  'amount': 'float64',
  'nameOrig': 'str',
  'oldbalanceOrg': 'float64',
  'newbalanceOrig': 'float64',
  'nameDest': 'str',
  'oldbalanceDest': 'float64',
  'newbalanceDest': 'float64',
  'isFraud': 'int64',
  'isFlaggedFraud': 'int64',
  'orig_balance_inconsistent': 'int8',
  'event_time': 'datetime64[ns, UTC]'},
 'notes': 'Simulated 30-day window anchored 2023-01-01 (synthetic). Balance inconsistencies kept as feature.',
 'label_distribution': {'0': 6354407, '1': 8213},
 'imbalance_ratio': 773.7}

In [9]:
u = pd.DataFrame({
    "event_time": df["event_time"],
    "event_subtype": df["type"].astype(str).str.lower(),
    "user_id": df["nameOrig"],
    "amount": df["amount"],
    "severity": np.where(df["isFraud"] == 1, 3, 0).astype("int8"),
    "label": df["isFraud"].astype("Int8"),
    "time_is_synthetic": True,
})
attr_cols = ["nameDest", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest",
             "newbalanceDest", "isFlaggedFraud", "orig_balance_inconsistent", "step"]
u[attr_cols] = df[attr_cols]
u = to_unified(u, source_dataset="paysim", event_domain="financial",
               event_type="mobile_txn", label_type="fraud", attributes_cols=attr_cols)
save_unified_part(u, "paysim")
u.head(3)

unified part -> /home/suryaguru/StudioProjects/sentinel_fusion_ai/data/unified/part_paysim.parquet (6,362,620 rows)


,event_id,event_time,event_domain,event_type,event_subtype,source_dataset,user_id,device_id,src_ip,dst_ip,...,amount,duration_s,bytes_in,bytes_out,severity,label,label_type,attack_technique,time_is_synthetic,attributes
0,paysim-0,2023-01-01 00:00:01.867478997+00:00,financial,mobile_txn,transfer,paysim,C883678948,<NA>,<NA>,<NA>,...,789419.02,NaN,NaN,NaN,0,0,fraud,<NA>,True,"{""nameDest"":""C1286084959"",""oldbalanceOrg"":0.0,..."
1,paysim-1,2023-01-01 00:00:02.045265332+00:00,financial,mobile_txn,cash_out,paysim,C1868809295,<NA>,<NA>,<NA>,...,161795.90,NaN,NaN,NaN,0,0,fraud,<NA>,True,"{""nameDest"":""C97730845"",""oldbalanceOrg"":0.0,""n..."
2,paysim-2,2023-01-01 00:00:03.842874082+00:00,financial,mobile_txn,transfer,paysim,C1886567481,<NA>,<NA>,<NA>,...,59531.87,NaN,NaN,NaN,0,0,fraud,<NA>,True,"{""nameDest"":""C998351292"",""oldbalanceOrg"":0.0,""..."
